In [1]:
# ==============================================================================
# CELL 1: INSTALL LIBRARIES & FORCE KERNEL RESTART
# ==============================================================================
# We pin specific, compatible versions to prevent conflicts during non-interactive runs.
! pip install -q -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ==============================================================================
# CELL 2: YOUR COMPLETE INFORMATION EXTRACTION SCRIPT
# ==============================================================================
# All imports go here, in the second cell.
import pandas as pd
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient # <-- Import the secrets client

# --- Authenticate using the Kaggle Secret ---
try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face.")
except:
    print("Could not find Hugging Face token in Kaggle Secrets. Please add it.")

# ==============================================================================
# 2. LOAD AND PREPARE YOUR DATASET
# ==============================================================================
print("\nStep 2: Loading and preparing the dataset...")

# --- Action Required: Update your file path and column names ---
# I am assuming your dataset is in the Kaggle input directory and has these column names.
# Please update them if they are different.
DATA_FILE_PATH = '/kaggle/input/rental-prices-with-ad-text/dataset_with_key_phrases (3).csv' # e.g., '/kaggle/input/rental-data/ads.csv'
AD_TEXT_COLUMN = 'body'  # The column with the full ad text
SOURCE_COLUMN = 'source'        # The column with the source (e.g., 'RentLingo')
CHUNK_SIZE = 1000
CHUNK_NUMBER = 5 # Kavithi: 0,1, Maheesha: 2, 3  Dushan:4,5 


try:
    print(f"\nStep 2: Loading data for Chunk #{CHUNK_NUMBER}...")
    df = pd.read_csv(DATA_FILE_PATH)
    
    # Normalize the source column (lowercase, strip whitespace) and filter for 'rentlingo'
    df[SOURCE_COLUMN] = df[SOURCE_COLUMN].str.lower().str.strip()
    rentlingo_ads_df = df[df[SOURCE_COLUMN] == 'rentlingo'].copy()
    
    # Get the list of raw advertisement texts to process
    all_advertisements = rentlingo_ads_df[AD_TEXT_COLUMN].dropna().tolist()
    print(f"Successfully loaded and filtered data. Found {len(all_advertisements)} ads from RentLingo to process.")

except FileNotFoundError:
    print(f"ERROR: The file '{DATA_FILE_PATH}' was not found. Please update the path.")
    all_advertisements = [] # Set to empty to prevent further errors

# --- SLICE THE DATA FOR THE CURRENT CHUNK ---
start_index = CHUNK_NUMBER * CHUNK_SIZE
end_index = start_index + CHUNK_SIZE
ads_for_this_chunk = all_advertisements[start_index:end_index]
print(f"Processing {len(ads_for_this_chunk)} ads from index {start_index} to {end_index}.")

2025-09-28 12:01:31.973189: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759060892.332897      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759060892.439342      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Successfully logged in to Hugging Face.

Step 2: Loading and preparing the dataset...

Step 2: Loading data for Chunk #600...
Successfully loaded and filtered data. Found 6912 ads from RentLingo to process.
Processing 5 ads from index 3000 to 3005.


In [3]:
# ==============================================================================
# 3. LOAD THE GEMMA MODEL (4-BIT QUANTIZED)
# ==============================================================================
if all_advertisements:
    print("\nStep 3: Loading the google/gemma-3-4b-it model...")
    text_gen_pipeline = pipeline(
    task="text-generation",
    model="google/gemma-3-4b-it",
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "quantization_config": {"load_in_4bit": True}
        }
    )

    print("Pipeline with quantized model loaded successfully!")


Step 3: Loading the google/gemma-3-4b-it model...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda:0


Pipeline with quantized model loaded successfully!


In [4]:
# ==============================================================================
# 4. DEFINE PROMPT AND PROCESSING LOGIC
# ==============================================================================
def create_extraction_prompt(advertisement_text):
    
    user_content = f"""
Analyze the rental advertisement text below and convert it into a single JSON object based on the provided schema and rules.
    
**RULES & SCHEMA:**
- For fields with predefined options (e.g., [provided, not]), choose the most appropriate option. If information is not mentioned, use "not" or "No".
- For rental price, extract the minimum and maximum values. If only one price is given, use it for both min and max.
- For unit types, extract the smallest and largest types mentioned (e.g., "studio" and "2 bedroom"). If only one is mentioned, use it for both.Srictly follow the naming pattern in JSON Schema. (e.g.,'2 Bedroom', 'Studio')
- Extract numbers only for numerical fields (e.g., price).
- for Outdoor_Spaces,
    - If 'Balcony, Deck, Patio' present together, then categorize as 'Combined'
    - If each 'Balcony', 'Deck' and 'Patio' present seperately, then categorize as 'Balcony', 'Patio' and 'Deck' respectively.
    - If non of the above words appear. then categorize as ' No Outdoor Space'
- for flooring,
    (1)If there are phrases their meaning expresses a wooden floor exists(e.g. Hard wood floors, wooen floors etc.), categorize as 'wood'
    (2)If there are phrases their meaning expresses a tiled floor exists(e.g. tiled floors etc.), categorize as 'tile'
    (3)If there are phrases their meaning expresses a carpet floor exists(e.g. Carpet etc.), categorize as 'carpet'
    - If all the above appear together, categorize them as 'wood+tile'((1) & (2)), 'carpet+wood'((3) & (1)) and 'tile+carpet'((2) & (3)) accordingly.
    - If any of the above cases appear, categorize it as 'other'.

**Required JSON Schema:**
{{
"rental_price": {{ "min": <number>, 
"max": <number> }},
"rental_type": "[Monthly, Weekly, other]",
"unit_details": {{ "min_type": "<Studio, 1 Bedroom, etc.>","max_type": "<1 Bedroom, 2 Bedroom, etc.>" }},
"amenities": {{
"Parking": "[provided, not]",
"Flooring": "[wood, carpet, tile, wood+tile, carpet+wood, tile+carpet, other]",
"Laundry": "[provided, not]",
"Dishwasher": "[provided, not]",
"Stainless_Appliances": "[provided, not]",
"Cable": "[included, not]",
"Internet": "[included, not]",
"Outdoor_Spaces": "[Balcony, Patio, Deck, Combined, No Outdoor Space]",
"AC": "[provided, not]",
"Pool": "[provided, not]",
"Fitness_Facilities": "[available, not]"
}},
"location": {{ "State": "<State>", "City": "<City>" }}
}}

**EXAMPLE**
    - **Input Text**: "This unit is located at 1421 Massachusetts Ave, Washington, 20005, DCMonthly rental rates range from $875 - $1114We have studio - 2 beds units available for rent
    Apartment features include:-Hard wood floors- Dishwasher- Heat Included- On-Site Laundry- Air conditioner- Controlled Access- On Bus Line- Water Included- Surface Parking- Pool- Stainless Appliances- Cable Included- Internet Included- Fitness facilities- Deck"
    - **JSON Output**:
    {{
    "rental_price": {{ "min": 875, "max": 1114 }},
    "rental_type": "Monthly",
    "unit_details": {{ "min_type": "Studio", "max_type": "2 Bedroom" }},
    "amenities": {{
    "Parking": "provided", 
    "Flooring": "wood", 
    "Laundry": "provided", 
    "Dishwasher": "provided", 
    "Stainless_Appliances": "provided", 
    "Cable": "included", 
    "Internet": "included", 
    "Outdoor_Spaces": "Deck", 
    "AC": "provided", "Pool": "provided", 
    "Fitness_Facilities": "available"   
    }},
    "location": {{ "State": "DC", "City": "Washington" }}
    }}
    
**EXAMPLE**
    - **Input Text**: "This unit is located at 202 West Longleaf Drive, Auburn, 36832, ALMonthly rental rates range from $1975We have  two - four beds units available for rent"
    - **JSON Output**:
    {{
    "rental_price": {{ "min": 1975, "max": 1975 }},
    "rental_type": "Monthly",
    "unit_details": {{ "min_type": "2 Bedroom", "max_type": "4 Bedroom" }},
    "amenities": {{
    "Parking": "not", 
    "Flooring": "other", 
    "Laundry": "not", 
    "Dishwasher": "not", 
    "Stainless_Appliances": "not", 
    "Cable": "not", 
    "Internet": "not", 
    "Outdoor_Spaces": "No Outdoor Space", 
    "AC": "not", 
    "Pool": "not", 
    "Fitness_Facilities": "not"   
    }},
    "location": {{ "State": "AL", "City": "Auburn"}}
    }}
    
**EXAMPLE**
    - **Input Text**: "This unit is located at 333 Hyde St, San Francisco, 94109, CAMonthly rental rates range from  $1390We have studio units available for rentApartment features include:-- Tile Floor- Balcony, Deck, Patio- On Bus Line- Refrigerator"
    - **JSON Output**:
    {{
    "rental_price": {{ "min": 1975, "max": 1975 }},
    "rental_type": "Monthly",
    "unit_details": {{ "min_type": "2 Bedroom", "max_type": "4 Bedroom" }},
    "amenities": {{
    "Parking": "not", 
    "Flooring": "tile", 
    "Laundry": "not", 
    "Dishwasher": "not", 
    "Stainless_Appliances": "not", 
    "Cable": "not", 
    "Internet": "not", 
    "Outdoor_Spaces": "combined", 
    "AC": "not", 
    "Pool": "not", 
    "Fitness_Facilities": "not"   
    }},
    "location": {{ "State": "CA", "City": "San Francisco"}}
    }}

    **Rental Advertisement Text to Analyze:**
    
    {advertisement_text}

    """

    return user_content

In [5]:
def process_rental_listing(model_output):
    # This is your price logic processing function
    unit_to_num = {"studio": 0, "1 bedroom": 1, "2 bedroom": 2, "3 bedroom": 3, "4 bedroom": 4, "5 bedroom": 5, "1 beds": 1, "2 beds": 2, "3 beds": 3, "4 beds": 4, "5 beds": 5}
    try:
        price_min = int(model_output.get("rental_price", {}).get("min"))
        price_max = int(model_output.get("rental_price", {}).get("max"))
    except ValueError:
        price_min = 0
        price_max = 0
        print("Error: Cannot convert non-numeric string to integer.")
    unit_min_str = model_output.get("unit_details", {}).get("min_type", "").lower()
    unit_max_str = model_output.get("unit_details", {}).get("max_type", "").lower()
    base_data = {}
    base_data.update(model_output.get("amenities", {}))
    base_data.update(model_output.get("location", {}))
    base_data["rental_type"] = model_output.get("rental_type")
    unit_min_num = unit_to_num.get(unit_min_str)
    unit_max_num = unit_to_num.get(unit_max_str)
    final_rows = []

    # --- START OF THE FIX ---
    # 1. Check if essential info (price and at least the min unit type) exists.
    #    If not, we can't process this ad, so we skip it.
    if price_min is None or unit_min_num is None:
        return [] 

    # 2. Check for a partial match in a range (e.g., min is valid but max is not).
    #    If the max unit type is not recognized, default to treating it as a single unit.
    if unit_min_num is not None and unit_max_num is None:
        unit_max_num = unit_min_num
    # --- END OF THE FIX ---
    
    if price_min == price_max or unit_min_num == unit_max_num:
        row = base_data.copy()
        row["Price"] = (price_min + price_max) / 2
        row["Unit_Type"] = unit_min_str
        row["Bedrooms"] = unit_min_num
        final_rows.append(row)
    else:
        num_unit_types = unit_max_num - unit_min_num + 1
        prices = np.linspace(price_min, price_max, num_unit_types)
        for i, unit_num in enumerate(range(unit_min_num, unit_max_num + 1)):
            row = base_data.copy()
            row["Price"] = prices[i]
            unit_str = [k for k, v in unit_to_num.items() if v == unit_num][0]
            row["Unit_Type"] = unit_str
            row["Bedrooms"] = unit_num
            final_rows.append(row)
    return final_rows

# ==============================================================================
# 5. EXECUTE BATCH PROCESSING
# ==============================================================================
if ads_for_this_chunk:
    print("\nStep 5: Starting parallel batch processing of all advertisements...")
    
    # Prepare all prompts for the batch
    all_prompts = []
    for ad_text in ads_for_this_chunk:
        prompt = create_extraction_prompt(ad_text)
        messages = [{"role": "system", "content": "You are a highly precise data extraction API. Your only function is to convert user-provided text into a single, valid JSON object according to the user's schema. You must not add any explanations or text outside of the JSON structure."},
                     {"role": "user", "content": prompt}]
        all_prompts.append(messages)
        
    # Execute the batch processing
    # Adjust batch_size based on your GPU memory. 8 is a safe start for Kaggle.
    outputs = text_gen_pipeline(all_prompts, batch_size=8, max_new_tokens=1024, return_full_text=False)
    print("Batch processing complete.")

    # ==============================================================================
    # 6. PROCESS RESULTS AND CREATE FINAL DATAFRAME
    # ==============================================================================
    print("\nStep 6: Processing results and creating the final DataFrame...")
if 'outputs' in locals():
    master_list = []
    for i, output in enumerate(outputs):
        response_text = output[0]['generated_text']
        try:
            json_start = response_text.find('{')
            json_end = response_text.rfind('}') + 1
            json_string = response_text[json_start:json_end]
            model_output = json.loads(json_string)
            
            # Apply your complex price logic
            processed_rows = process_rental_listing(model_output)
            master_list.extend(processed_rows)
        except (json.JSONDecodeError, IndexError):
            print(f"Warning: Could not parse JSON for ad #{i+1}. Skipping.")

    # Create the final comprehensive DataFrame
    df_chunk_final = pd.DataFrame(master_list)
    
    # ==============================================================================
    # 7. SAVE THE FINAL DATASET
    # ==============================================================================
    print("\nStep 7: Saving the final structured dataset...")
    output_path = f'/kaggle/working/structured_data_chunk_{CHUNK_NUMBER}.csv'
    df_chunk_final.to_csv(output_path, index=False)
    
    print(f"\n✅ Success! Chunk {CHUNK_NUMBER} is saved at: {output_path}")


Step 5: Starting parallel batch processing of all advertisements...
Batch processing complete.

Step 6: Processing results and creating the final DataFrame...

Step 7: Saving the final structured dataset...

✅ Success! Chunk 600 is saved at: /kaggle/working/structured_data_chunk_600.csv
